In [ ]:
from google.colab import drive
import os
import torch

# 1. Check GPU & PyTorch (Sanity Check)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
!nvidia-smi

# 2. Mount Drive
drive.mount('/content/drive')

# 3. Configuration
# Use your specific repo here. 
REPO_URL = "https://github.com/fabianandresgrob/gaussian-splatting.git"
REPO_BRANCH = "view-selection"
WORKING_DIR = "/content/gsplat_workspace"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

# 4. Create Workspace & Clone
os.makedirs(WORKING_DIR, exist_ok=True)
os.chdir(WORKING_DIR)

if not os.path.exists(REPO_NAME):
    print(f"--- Cloning {REPO_NAME} ---")
    !git clone {REPO_URL} --recursive
    os.chdir(REPO_NAME)
    !git checkout {REPO_BRANCH}
else:
    print(f"--- Repository exists. Pulling latest ---")
    os.chdir(REPO_NAME)
    !git pull
    !git checkout {REPO_BRANCH}
    !git submodule update --init --recursive

REPO_PATH = os.getcwd()
print(f"SUCCESS: Repo ready at {REPO_PATH}")

In [ ]:
# --- CELL 4: EXECUTION ---
import setup_utils
import experiment_lib
import pandas as pd
import os

# 1. Install Dependencies & Compile Extensions (Repo path from Cell 1)
# This uses the logic you provided: patches files and runs setup.py
setup_utils.setup_environment(os.getcwd())

In [ ]:
# 2. Define Experiments
DATA_ROOT = "/content/drive/MyDrive/Scannet++/data/scenes/data" # <--- UPDATE YOUR PATH
OUTPUT_ROOT = "/content/drive/MyDrive/3DGS_Results" # <--- UPDATE YOUR PATH

SCENES = ["0c5385e84b"] 
STRATEGIES = ["random", "fixed_prob"] 
SEEDS = [0, 1, 2, 3, 4]

# Training configuration
TRAINING_CONFIG = {
    "iterations": 30000,
    "test_iterations": [1000, 3000, 7000, 15000, 30000],
    "save_iterations": [30000],
    "data_device": "cuda",
}

# Strategy-specific configurations (optional)
STRATEGY_CONFIGS = {
    "random": "{}",
    "fixed_prob": '{"temperature": 1.0, "distance_weight": 0.5, "diversity_weight": 0.5}',
    "epoch_based": '{"penalty_strength": 1.0}',
    "clustering": '{"n_clusters": 10, "temperature": 1.0}',
    "no_replace": "{}",
}

# 3. Run Experiments
current_repo_path = os.getcwd()

for scene in SCENES:
    scene_path = os.path.join(DATA_ROOT, scene, "dslr")
    
    for strategy in STRATEGIES:
        exp_name = f"{scene}_{strategy}"
        view_config = STRATEGY_CONFIGS.get(strategy, "{}")
        
        for seed in SEEDS:
            experiment_lib.run_training(
                repo_path=current_repo_path,
                data_path=scene_path,
                output_dir=OUTPUT_ROOT,
                strategy=strategy,
                seed=seed,
                exp_name=exp_name,
                iterations=TRAINING_CONFIG["iterations"],
                test_iterations=TRAINING_CONFIG["test_iterations"],
                save_iterations=TRAINING_CONFIG["save_iterations"],
                data_device=TRAINING_CONFIG["data_device"],
                view_selection_config=view_config,
            )

In [ ]:
# 4. Final Report
results = []
for scene in SCENES:
    for strategy in STRATEGIES:
        rep = experiment_lib.aggregate_results(OUTPUT_ROOT, scene, strategy)
        if rep: results.append(rep)

df = pd.DataFrame(results)
print("\n=== FINAL REPORT ===")
print(df)
df.to_csv(os.path.join(OUTPUT_ROOT, "final_experiment_report.csv"), index=False)

Results Directory Structure
===========================
```
/content/drive/MyDrive/3DGS_Results/       <-- Your OUTPUT_ROOT_DRIVE
│
├── 0c5385e84b_random/                     <-- Experiment Folder ({scene}_{strategy})
│   ├── seed_0/                            <-- Run Folder (Per Seed)
│   │   ├── point_cloud/
│   │   │   └── iteration_30000/
│   │   │       └── point_cloud.ply        <-- Final trained Gaussian model
│   │   ├── eval/
│   │   │   ├── 00000.png                  <-- Rendered test image 0
│   │   │   ├── 00001.png
│   │   │   └── ...
│   │   ├── combine/
│   │   │   ├── 00000.png                  <-- Side-by-side: Render vs. GT
│   │   │   └── ...
│   │   ├── metrics_history.json           <-- Training curve data (Loss/PSNR over time)
│   │   ├── final_results.json             <-- Final averaged metrics for this seed
│   │   ├── console_log.txt                <-- Full training log output
│   │   ├── cameras.json                   <-- Camera parameters used
│   │   ├── cfg_args                       <-- Arguments used for this run
│   │   ├── chkpnt30000.pth                <-- PyTorch checkpoint
│   │   └── events.out.tfevents...         <-- Tensorboard logs
│   │
│   ├── seed_1/
│   │   └── ... (Same structure)
│   └── ...
│
├── 0c5385e84b_fixed_prob/                 <-- Next Experiment
│   └── ...
│
└── experiment_summary.csv                 <-- Final aggregated report of all runs
```